In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import myfunction as mf
path_data_raw = "C:/Users/dell/OneDrive/file/"
path_country_nc = "C:/Users/dell/OneDrive/file/nc"
path_csv = "C:/Users/dell/OneDrive/file/csv/"
path_one_spdb = 'C:/Users/dell/OneDrive/file/SPDB/'
drive_letter = 'E:'

path_pre = drive_letter + "/wyy/code_project/running_outcome/final_data/SPDB/part0_treat/pretreatment/"
path_match = drive_letter + "/wyy/code_project/running_outcome/final_data/SPDB/part0_treat/match/"

path_2_preanalysis_data = drive_letter + "/wyy/code_project/running_outcome/final_data/SPDB/part2_analysis/preanalysis/"
path_2_preanalysis_fig = drive_letter + "/wyy/code_project/running_outcome/final_fig/SPDB/part2_analysis/preanalysis/"
path_3_sw_forecast = drive_letter + "/wyy/code_project/running_outcome/final_data/SPDB/part3_forecast/sw_forecast/"
path_temp = drive_letter + "/wyy/code_project/running_outcome/final_data/SPDB/part2_analysis/temp/"

path_sw_rfecv_data = drive_letter + '/wyy/code_project/running_outcome/final_data/SPDB/part3_forecast/sw_forecast/'
path_lrsw_rfecv_data = drive_letter + '/wyy/code_project/running_outcome/final_data/SPDB/part3_forecast/lrsw_forecast/'


meta_name = "meta_data.csv"




In [ ]:

from factor_analyzer import FactorAnalyzer
from openpyxl import Workbook
from openpyxl.styles import PatternFill
from openpyxl.utils.dataframe import dataframe_to_rows
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats




def forward_select_vif(df_features_raw, one_var, max_vif=10):
    """这个跟下面的区别在于这个只能指定一个初始变量"""
    selected_vars = [one_var]
    remaining_vars = [var for var in df_features_raw.columns if var != one_var]
    vif_data = pd.DataFrame(columns=['variables', 'VIF'])

    while remaining_vars:
        temp_vif_data = pd.DataFrame(columns=['variables', 'VIF'])
        for var in remaining_vars:
            temp_df = df_features_raw[selected_vars + [var]]
            temp_vif = [variance_inflation_factor(temp_df.values, i)
                        for i in range(temp_df.shape[1])]

            if all(vif < max_vif for vif in temp_vif[:-1]):  # 排除添加的当前变量的VIF

                new_row = pd.DataFrame({'variables': [var], 'VIF': [temp_vif[-1]]})
                temp_vif_data = pd.concat([temp_vif_data, new_row], ignore_index=True)

        temp_vif_data['VIF'] = pd.to_numeric(temp_vif_data['VIF'], errors='coerce')

        temp_vif_data = temp_vif_data[temp_vif_data['VIF'] < max_vif]
        if temp_vif_data.empty:
            break

        next_var = temp_vif_data.loc[temp_vif_data['VIF'].idxmin(), 'variables']
        selected_vars.append(next_var)
        remaining_vars.remove(next_var)

    final_df = df_features_raw[selected_vars]
    vif_data['variables'] = final_df.columns
    vif_data['VIF'] = [variance_inflation_factor(final_df.values, i) 
                       for i in range(final_df.shape[1])]

    return vif_data


def forward_select_vif2(df_features_raw, list_var, max_vif=10):
    """通过vif_select_var调用的话用sem开头的数据即可  直接调用的话注意 list_var是初始保留的变量组
        初始变量组只有一个元素无前置条件 但是多个得先通过count_vif函数"""
    remaining_vars = [var for var in df_features_raw.columns if var not in list_var]
    print(len(remaining_vars))
    vif_data = pd.DataFrame(columns=['variables', 'VIF'])

    while remaining_vars:
        temp_vif_data = pd.DataFrame(columns=['variables', 'VIF'])
        for var in remaining_vars:
            temp_df = df_features_raw[list_var + [var]]
            temp_vif = [variance_inflation_factor(temp_df.values, i)
                        for i in range(temp_df.shape[1])]

            if all(vif < max_vif for vif in temp_vif[:-1]):  # 排除添加的当前变量的VIF

                new_row = pd.DataFrame({'variables': [var], 'VIF': [temp_vif[-1]]})
                temp_vif_data = pd.concat([temp_vif_data, new_row], ignore_index=True)

        temp_vif_data['VIF'] = pd.to_numeric(temp_vif_data['VIF'], errors='coerce')

        temp_vif_data = temp_vif_data[temp_vif_data['VIF'] < max_vif]
        if temp_vif_data.empty:
            break

        next_var = temp_vif_data.loc[temp_vif_data['VIF'].idxmin(), 'variables']
        list_var.append(next_var)
        remaining_vars.remove(next_var)


    final_df = df_features_raw[list_var]
    vif_data['variables'] = final_df.columns
    vif_data['VIF'] = [variance_inflation_factor(final_df.values, i) 
                       for i in range(final_df.shape[1])]

    return vif_data


def count_vif(df_features_raw, max_vif=10):
    """迭代计算输入的dataframe VIF 每次迭代去除VIF最大的变量 直到所有变量的VIF小于10
        返回一个dataframe 里面包含两列 variables VIF"""
    vif_data = pd.DataFrame()
    vif_data["variables"] = df_features_raw.columns

    while True:

        vif_data["VIF"] = [variance_inflation_factor(df_features_raw.values, i) 
                        for i in range(len(df_features_raw.columns))]

        if vif_data['VIF'].max() > max_vif:
            max_vif_feature = vif_data.loc[vif_data['VIF'].idxmax(), 'variables']
            df_features_raw = df_features_raw.drop(max_vif_feature, axis=1)
            vif_data = vif_data[vif_data['variables'] != max_vif_feature]
            print('Remove var:',max_vif_feature)
        else:
            break
    print(vif_data.shape)
    return vif_data


def count_fa(df, level_name, flash_excel="NO", feature_type="standardization"):
    """根据df的数据 使用因子分析 识别潜在变量 level_name命名输出文件 flash_excel等于NO时不更改excel
        excel是给人看的 里面会标注重要变量  feature_type命名输出文件
        这个函数直接输出文件 返回的只有okk"""
    df_meta = pd.read_csv(path_data_raw + meta_name, encoding="utf-8")

    var_to_meaning = dict(zip(df_meta['var_name'], df_meta['meaning']))
    fa = FactorAnalyzer(rotation='varimax')
    fa.fit(df)
    loadings = fa.loadings_
    eigen_values, vectors = fa.get_eigenvalues()
    num_factors = sum(eigen_values > 1)
    print("num_factors:", num_factors)
    adjusted_eigen_values = list(eigen_values[:num_factors])#  + [None] * (len(loadings[0]) - num_factors)
    fa = FactorAnalyzer(n_factors=num_factors, rotation='varimax')
    fa.fit(df)
    loadings = fa.loadings_
    explained_variance = fa.get_factor_variance()
    explained_variance_df = pd.DataFrame(explained_variance, 
                                        index=['SS Loadings','Proportion Var','Cumulative Var'], 
                                        columns=['Factor'+str(i+1) for i in range(num_factors)])
    explained_variance_df.loc['Eigenvalues'] = adjusted_eigen_values
    loadings_df = pd.DataFrame(loadings, index=df.columns, 
                            columns=['Factor'+str(i+1) for i in range(num_factors)])
    loadings_df = pd.concat([explained_variance_df, loadings_df])
    scores = fa.transform(df)
    loadings_df['Meaning'] = loadings_df.index.map(var_to_meaning)
    loadings_df = loadings_df.reset_index().rename(columns={'index': 'vars'})
    cols = loadings_df.columns.tolist()
    cols.insert(0, cols.pop(cols.index('Meaning')))
    loadings_df = loadings_df.reindex(columns=cols)

    scores = fa.transform(df)

    scores_df = pd.DataFrame(scores, columns=['Factor'+str(i+1) for i in range(num_factors)])
    loadings_df.to_csv(path_temp + 'fa_loadings_' + feature_type+ '_' + level_name +'.csv',index=False)
    scores_df.to_csv(path_2_preanalysis_data + 'fa_scores_' + feature_type+ '_' + level_name +'.csv',index=False)
    if flash_excel == "NO":
        print("pass")
        pass
    else:
        print("flash excel")
        df_e = pd.read_csv(path_temp + 'fa_loadings_' + feature_type+ '_' + level_name +'.csv')

        cols = ['Factor'+str(i+1) for i in range(num_factors)]

        color_fill = PatternFill(start_color="90EE90",
                                    end_color="90EE90",
                                    fill_type="solid")
        wb = Workbook()
        ws = wb.active

        for r in dataframe_to_rows(df_e, index=False, header=True):
            ws.append(r)

        cols_index = [df_e.columns.get_loc(c) + 1 for c in cols]
        for row in ws.iter_rows(min_row=2, min_col=min(cols_index), max_col=max(cols_index), max_row=len(df_e)+1):
            values = [cell.value for cell in row]
            max_abs_value = max(values, key=abs)
            if abs(max_abs_value) >= 0.3:

                original_max_value = max(values, key=lambda x: abs(x) == abs(max_abs_value))
                for cell in row:
                    if cell.value == original_max_value:
                        cell.fill = color_fill
        wb.save(path_2_preanalysis_data + 'fa_' + feature_type+ '_' + level_name +'.xlsx')
    return num_factors


def print_equation(df, str_var='NO'):
    """快速生成model中潜在变量的公式 方便写代码
        这个函数直接输出文本 返回的只有okk"""
    df = df.iloc[4:]
    df = df.drop("Meaning", axis=1)
    df.set_index('vars', inplace=True)
    dic_var = {col: [] for col in df.columns}

    for index, row in df.iterrows():
        max_val = row.abs().max()
        if max_val >= 0.3:
            max_col = row.abs().idxmax()
            dic_var[max_col].append(index)

    if str_var != 'NO':
        new_dic_var = {}
        for i, (key, value) in enumerate(dic_var.items()):
            new_key = f'{str_var}_{i}'
            new_dic_var[new_key] = value
        dic_var = new_dic_var

    for key, value in dic_var.items():
        str_value = ' + '.join(value)
        print(f'{key} =~ {str_value}')

    return "okk"



def count_variance(df_sem_data):
    """计算所有变量的方差  返回的是一个df"""
    print(df_sem_data.shape)
    mean = df_sem_data.mean()
    std_dev = df_sem_data.std()
    variance = df_sem_data.var()


    df_results = pd.DataFrame({
        'var': df_sem_data.columns,
        'mean': mean.values,
        'SD': std_dev.values,
        'variance': variance.values
    })


    min_variance = df_results["variance"].min()
    max_variance = df_results["variance"].max()
    print(min_variance, max_variance)
    print(max_variance/min_variance)

    return df_results



def vif_select_var(df_o, path_2_preanalysis_data, path_rfecv_data, str_describe, cv_marker='' , int_max_vif=10, int_mlr=3, int_ml=10):
    """前置条件通过R语言的packfor包的sfs+mlr算法筛选变量和rfecv_imp.ipynb筛选变量"""
    df_data = df_o.copy()
    df_mlr = pd.read_csv(path_2_preanalysis_data + 'mlr_'+str_describe+'.csv')
    df_mlr = df_mlr.sort_values(by='R2', ascending=False)





    df_rfecv = pd.read_csv(path_rfecv_data + str_describe + '_merge_rfecv'+cv_marker+'.csv')

    min_features = df_rfecv.loc[0, 'min_features']
    max_model = df_rfecv.loc[0, 'model']
    max_score = round(df_rfecv.loc[0, 'mean_test_score'],3)
    print(f'model:{max_model} min_feature:{min_features} score:{max_score}')
    df_ml  =pd.read_csv(path_rfecv_data + str_describe + '_rfecv_features_'+max_model+'cv'+cv_marker+'.csv')
    df_ml = df_ml[df_ml['Rank']==1]





    list_mlr_var0 = df_mlr['variables'].values
    list_ml_var0 = df_ml['Feature'].values
    list_re_imp0 = list(set((set(list_mlr_var0) | set(list_ml_var0))))



    list_mlr_var1 = df_mlr['variables'][df_mlr['R2']>=(int_mlr/100)].values
    print(f'mlr lrsw:{list_mlr_var1}')
    list_ml_var1 = df_ml['Feature'][df_ml['Importance']>=int_ml/100].values
    print(f'ml lrsw:{list_ml_var1}')
    list_re_imp1 = list(set((set(list_mlr_var1) | set(list_ml_var1))))
    print(f'select var:{len(list_re_imp1)}  var:{list_re_imp1}')



    list_mlr_var2 = df_mlr['variables'][df_mlr['R2']<(int_mlr/100)].values
    list_ml_var2 = df_ml['Feature'][df_ml['Importance']<int_ml/100].values
    list_re_imp2 = list(set((set(list_mlr_var2) | set(list_ml_var2))))
    print(f'other var num:{len(list_re_imp2)}')


    df_data0 = df_data[list_re_imp0]
    df_data1 = df_data[list_re_imp1]


    df_vif_sbs = count_vif(df_data1,int_max_vif)
    list_start_var = df_vif_sbs['variables'].tolist()
    print(f'start var:{list_start_var}')

    df_vif_sfs = forward_select_vif2(df_data0,list_start_var,max_vif=int_max_vif)
    list_o = df_data0.columns.to_list()
    list_s = df_vif_sfs["variables"].to_list()
    list_remove  = [i for i in list_o if i not in list_s]
    print(f'SFS select var:{list_s}')
    print(f'var num:{len(list_s)}  remove num:{len(list_remove)}')
    return df_vif_sfs



In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

df_raw = pd.read_csv(path_match + 'sw_analysis_s1.csv')

list_select = df_raw.columns.tolist()



list_select.remove('value')
y = df_raw['value']
X = df_raw[list_select]

if not X.empty:
    X = sm.add_constant(X)  # 添加常数项
    model = sm.OLS(y, X).fit()
    print(model.summary())
else:
    print("X is empty. Check your data selection.")



In [13]:
path_sw_rfecv_data = drive_letter + '/wyy/code_project/running_outcome/final_data/SPDB/part3_forecast/sw_forecast/'
str_describe = 'sw'
df_data = pd.read_csv(path_match + 'sw_match_treat.csv')
df_vif_sfs = vif_select_var(df_data, path_2_preanalysis_data, path_sw_rfecv_data, str_describe)
print(df_vif_sfs.head())
df_vif_sfs.to_csv(path_2_preanalysis_data +str_describe+'_vif_sfs.csv', index=False)

model:RF min_feature:41 score:0.83
mlr lrsw:['TOTALS_CO2_E' 'po_m_w' 'GDP']
ml lrsw:['GDP']
select var:3  var:['GDP', 'po_m_w', 'TOTALS_CO2_E']
other var num:46
(3, 2)
start var:['GDP', 'po_m_w', 'TOTALS_CO2_E']
44
SFS select var:['GDP', 'po_m_w', 'TOTALS_CO2_E', 'cfc', 'HCFC', 'sp', 'sshf', 'SF6', 'e', 'solubility', 'TNR_Ship_CO2_E', 'ctp', 'log_Kaw', 'SWD_INC_CO2_E', 'log_pKa', 'nships_smoothed', 'HFC', 'str', 'cl', 'log_Koc', 'po_chain', 'i10fg', 'tcsw', 'global_salinity', 'lict', 'z', 'ssr']
var num:27  remove num:20
      variables       VIF
0           GDP  9.671165
1        po_m_w  4.297970
2  TOTALS_CO2_E  9.950030
3           cfc  2.918589
4          HCFC  1.923122


In [ ]:
path_lrsw_rfecv_data = drive_letter + '/wyy/code_project/running_outcome/final_data/SPDB/part3_forecast/lrsw_forecast/'
str_describe = 'lr_sw'
df_data = pd.read_csv(path_match + 'lr_sw_match_treat_full.csv')
df_vif_sfs = vif_select_var(df_data, path_2_preanalysis_data, path_lrsw_rfecv_data, str_describe)
print(df_vif_sfs.head())
df_vif_sfs.to_csv(path_2_preanalysis_data +str_describe+ '_vif_sfs.csv', index=False)

model:LGBM min_feature:52 score:0.74
mlr lrsw:['po_chain' 'sw_value' 'e' 'logD7_4']
ml lrsw:['sw_value']
select var:4  var:['e', 'po_chain', 'sw_value', 'logD7_4']
other var num:53
(4, 2)
start var:['e', 'po_chain', 'sw_value', 'logD7_4']
50
SFS select var:['e', 'po_chain', 'sw_value', 'logD7_4', 'sp_length', 'HCFC', 'sp_troph', 'cfc', 'organ_liver', 'SWD_INC_CO2_E', 'sp', 'wrap_consumption', 'solubility', 'tcsw', 'str', 'HFC', 'sshf', 'TNR_Ship_CO2_E', 'global_salinity', 'licd']
var num:20  remove num:34
   variables       VIF
0          e  4.141679
1   po_chain  3.964321
2   sw_value  2.336923
3    logD7_4  3.097002
4  sp_length  1.168479


In [ ]:
str_describe = 'lr_sw'
sw_mark = 'sw'
df_vif_lr = pd.read_csv(path_2_preanalysis_data + 'lr_sw_vif_sfs.csv')
df_vif_sw = pd.read_csv(path_2_preanalysis_data + 'sw_vif_sfs.csv')

list_vars_lr = df_vif_lr['variables'].tolist()
list_vars_sw = df_vif_sw['variables'].tolist()


print('lr_all:', len(list_vars_lr))
print('sw_all:', len(list_vars_sw))


meta_data = pd.read_csv("C:\\Users\\dell\\OneDrive\\file\\meta_data.csv")

list_hue_var = meta_data['var_name'][meta_data['var_type3']==0].tolist()
list_env_var = meta_data['var_name'][meta_data['var_type3']==1].tolist()
list_pfas_var = meta_data['var_name'][meta_data['var_type3']==2].tolist()
list_sp_var = meta_data['var_name'][meta_data['var_type3']==3].tolist()


list_lr_hue = list(set(list_vars_lr) & set(list_hue_var))
list_lr_env = list(set(list_vars_lr) & set(list_env_var))
list_lr_pfas = list(set(list_vars_lr) & set(list_pfas_var))
list_lr_sp = list(set(list_vars_lr) & set(list_sp_var))

list_sw_hue = list(set(list_vars_sw) & set(list_hue_var))
list_sw_env = list(set(list_vars_sw) & set(list_env_var))
list_sw_pfas = list(set(list_vars_sw) & set(list_pfas_var))

print(f'lr hue:{list_lr_hue}')
print(f'lr env:{list_lr_env}')
print(f'lr pfas:{list_lr_pfas}')
print(f'lr sp:{list_lr_sp}')

print(f'sw hue:{list_sw_hue}')
print(f'sw env:{list_sw_env}')
print(f'sw pfas:{list_sw_pfas}')


lr_all: 27
sw_all: 27
lr hue:['HFC', 'HCFC', 'TNR_Ship_CO2_E', 'sw_value', 'wrap_consumption', 'SWD_INC_CO2_E', 'CF4']
lr env:['e', 'str', 'sp', 'cl', 'tcsw', 'licd', 'global_salinity', 'sro', 'ssro', 'cfc', 'sshf']
lr pfas:['density', 'log_Koil_w', 'logD7_4', 'po_chain', 'solubility']
lr sp:['sp_troph', 'sp_length', 'organ_liver', 'organ_muscle']
sw hue:['TOTALS_CO2_E', 'GDP', 'nships_smoothed', 'SF6', 'HFC', 'HCFC', 'TNR_Ship_CO2_E', 'SWD_INC_CO2_E']
sw env:['z', 'e', 'str', 'sp', 'cl', 'ctp', 'tcsw', 'i10fg', 'global_salinity', 'lict', 'ssr', 'cfc', 'sshf']
sw pfas:['log_pKa', 'log_Koc', 'log_Kaw', 'po_m_w', 'po_chain', 'solubility']
